In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


loader=TextLoader("kb.txt", encoding="utf-8")
loaded_docs=loader.load()


splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
splitted_docs=splitter.split_documents(loaded_docs)


In [5]:
print(f"loaded {len(splitted_docs)} documents")

loaded 7 documents


In [10]:
from langchain_openai import OpenAIEmbeddings

embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore=FAISS.from_documents(documents=splitted_docs,embedding=embeddings)


In [15]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-4o-mini",temperature=0.0)

In [17]:
retriever=vectorstore.as_retriever(search_kwargs={"k":3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



In [19]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful AI assistant answering questions based only on the provided context.

Instructions:
1. Answer ONLY using the provided context.
2. If the answer is not present in the context, do NOT guess or use your own knowledge.
3. Instead, politely say:
   "I couldn't find that information in the uploaded documents."
4. Encourage the user to ask another question related to the uploaded documents.
5. Keep your answers clear, concise, and accurate.

Context:
{context}
"""
    ),
    ("human", "{question}")
])

In [26]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain=({
    "context":retriever| format_docs,
    "question":RunnablePassthrough()
})|prompt|llm|StrOutputParser()

'Chapter 4 is about modern machine learning applications and builds directly on the fundamentals of neural networks. It includes a recap and extension of concepts such as layers, weights, activation functions, and backpropagation, which are explained in Chapter 1. Additionally, it discusses convolutional networks for images. \n\nFeel free to ask another question related to the uploaded documents!'

In [31]:
# vector_rag_naive.py
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA


# 1. Load the raw text
with open("kb.txt", "r") as f:
    raw_text = f.read()

# 2. Naive chunking — no awareness of chapter/section boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # characters, deliberately arbitrary
    chunk_overlap=50,
)
chunks = splitter.split_text(raw_text)
print(f"Naive chunking produced {len(chunks)} chunks")

# 3. Embed and index
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(chunks, embeddings)

# 4. Retrieval + QA
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
)

# 5. Run your eval queries
queries = [
    "How does backpropagation relate to router traffic-shaping?",
    "How are CNN filters connected to what was covered earlier in the book?",
    "Where are trained model weights stored and how are they looked up efficiently?",
    "What does 'network' mean in this book?",
]

for q in queries:
    result = qa.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print("Retrieved chunks (first 80 chars each):")
    for doc in result["source_documents"]:
        print(f"  - {doc.page_content[:80]}...")

Naive chunking produced 16 chunks

Q: How does backpropagation relate to router traffic-shaping?
A: Backpropagation and router traffic-shaping are distinct processes. Backpropagation is a method used in training neural networks, where the error from the network's output is propagated backward to adjust weights and improve predictions. In contrast, router traffic-shaping involves adjusting routing weights based on observed congestion patterns, which is a simpler mechanism and does not involve the same learning or weight-adjustment process as backpropagation. Therefore, while both involve adjustments based on observed data, they operate in different contexts and with different underlying mechanisms.
Retrieved chunks (first 80 chars each):
  - 3.3 Note on Learning Systems

Some modern network routers now use adaptive traff...
  - 1.3 Training via Backpropagation

Training works by comparing the network's outp...
  - 1.2 Layers and Activation

A typical network has an input layer, one or m